In [429]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [430]:
df = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')

In [431]:
df.head()

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,0.39006,0.10963,0.55097,-0.56104,1,NaN,approve,0,7.9,NaN
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,0.03265,-0.40256,0.36218,0.86583,1,NaN,approve,0,5.5,NaN
2,CUST_8ZI3LCBZ0W,ACC_AF53381QSC0L,27,0,20326.87,720.0,25,73.61,492.57,4.6,...,-0.15637,0.57818,0.28902,-2.19864,1,NaN,approve,0,7.2,NaN
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703.0,17,47.53,204.18,25.3,...,-1.02145,0.63908,-0.89190,-0.81592,1,NaN,approve,0,4.4,NaN
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,NaN,594.0,13,99.95,734.09,12.8,...,-0.65771,0.08020,0.17606,0.86739,1,NaN,approve,0,4.9,NaN


## A. Qualité des données / nettoyage logique

### 1.1 Valeurs impossibles détectées

In [432]:
# suppression des âges négatifs
df = df[df['age'] >= 0]
# suppression des dates de création de compte négatives
df = df[df["tenure_months" ]>= 0]
# suppression des revenus négatifs
df = df[df["annual_income_eur"] >= 0]
# suppression des revenus négatifs
df = df[df["avg_amount_30d_eur"] >= 0]

### 1.2 Corrections de typologie nécessaires

In [433]:
# Conversions de types
df['is_new_device'] = df['is_new_device'].astype('int64')
df['postal_code'] = df['postal_code'].astype('string')
df['days_since_last_login'] = df['days_since_last_login'].astype('int64')
df['signup_source'] = df['signup_source'].astype('object')

### 1.3 suppression doublons inutiles

In [434]:
# suppression de 680 lignes avec des customer_id en double ( très peu de changement donc variation)
df = df.drop_duplicates(subset=['customer_id'], keep='first')

### 1.4 doublons stricts

In [435]:
df = df.drop_duplicates()

## B. Data leakage connu

In [436]:
colonnes_a_supprimer = [
    'chargeback_resolution_time_days',
    'post_event_status_code'
]

df = df.drop(columns=colonnes_a_supprimer)

## C. Identifiants et colonnes inutiles

In [437]:
# Suppression des colonnes avec trop de NaN (>90%)
cols_to_drop = ["legacy_partner_score", "partner_risk_indicator"]
df = df.drop(columns=cols_to_drop)

In [438]:
# suppression de 680 lignes avec des customer_id en double ( très peu de changement donc variation)
df = df.drop_duplicates(subset=['customer_id'], keep='first')

In [439]:
df = df.drop("referrer_code", axis=1) # trop granulaire
df = df.drop("postal_code", axis=1) # trop granulaire : risque overfitting
df = df.drop("account_id", axis=1) # inutile
df = df.drop("signup_date", axis=1) # pas d'informations facilement exploitable

In [440]:
# Créer la colonne has_second_email (1 si secondary_email existe, 0 sinon)
df['has_second_email'] = df['secondary_email'].notna().astype(int)

# Supprimer la colonne secondary_email
df = df.drop(columns=['secondary_email'])

## D. Création de features basiques

In [441]:
# Créer les flags pour TOUTES les variables avec missingness informatif
df['is_missing_last_ticket_subject'] = df['last_ticket_subject'].isna().astype(int)
df['is_missing_max_amount_30d_eur'] = df['max_amount_30d_eur'].isna().astype(int)

In [442]:
# création de combinaison nouvelles et logiques 

# Interaction binaire × numérique
df["is_new_device_x_num_devices"] = df["is_new_device"] * df["num_devices_30d"]

# Interaction binaire × binaire
df["is_vpn_x_ip_risk"] = df["is_vpn"] * df["ip_risk_z"]

## E. imputations ne dépendant pas de statistiques

In [443]:
df['max_amount_30d_eur'] = df['max_amount_30d_eur'].fillna(0)

In [444]:
df["ip_risk_z"] = df["ip_risk_z"].fillna(0)

## Début : Suppression temporaire des features catégorielles à conseerver mais non encodés
## Je les rajouterai petit à petit

In [445]:
# Nombre total de lignes
n_rows = len(df)

# Calcul du nombre et du pourcentage de NaN
missing_df = pd.DataFrame({
    'nb_null': df.isnull().sum(),
    'percent_null': df.isnull().sum() / n_rows * 100
})

# Trier par % décroissant
missing_df = missing_df.sort_values(by='percent_null', ascending=False)
missing_df.head(8)

,nb_null,percent_null
region,38932,28.292988
credit_score,6882,5.001344
device_trust_z,5488,3.988285
customer_note,4132,3.002842
occupation,4073,2.959965
is_vpn_x_ip_risk,4069,2.957058
last_ticket_subject,4037,2.933802
merchant_category,2714,1.972341


In [446]:
fill_values = {
    'occupation': 'Missing',
    'merchant_category': 'Missing',
    'last_ticket_subject': 'No_ticket',
    'customer_note': 'No_note'
}

df.fillna(value=fill_values, inplace=True)

# Liste des colonnes à supprimer
caté_a_tester = [
    "country",
    "occupation",
    "last_ticket_subject",
    "customer_note",
    "payment_method",
    "merchant_category",
    "signup_source",
    "os",
    "browser",
    "device_type",
    "channel",
    "plan_type",
    "city",
    "customer_id"
]

# Supprimer les colonnes
df = df.drop(columns=caté_a_tester)

In [467]:
df.to_csv("data_propre_non_scale.csv", index=False)

# -----------------------------------------------
#                            TRAIN TEST SPLIT
# -----------------------------------------------

In [447]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# Option si tu as category_encoders
from category_encoders import TargetEncoder

In [448]:
X = df.drop(columns=['target_is_fraud'])
y = df['target_is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Multicolinéarité

In [449]:
# Liste des colonnes à supprimer
col_vif_trop_fort = [
    'terms_accepted_flag',
    'credit_score',
    'income_log',
    'avg_amount_30d_eur',
    'credit_score_norm',
    'income_estimate_alt_eur',
    'max_to_avg_ratio',
    'age',
    'tx_amount_total_30d_eur'
]

# Supprimer les colonnes dans X_train et X_test
X_train = X_train.drop(columns=col_vif_trop_fort)
X_test = X_test.drop(columns=col_vif_trop_fort)

## A. Encodege Catégorielle 

In [450]:
#median_credit_score = X_train['credit_score'].median()

#X_train['credit_score'] = X_train['credit_score'].fillna(median_credit_score)
#X_test['credit_score'] = X_test['credit_score'].fillna(median_credit_score)


#  device_trust_z → 0
X_train['device_trust_z'] = X_train['device_trust_z'].fillna(0)
X_test['device_trust_z'] = X_test['device_trust_z'].fillna(0)


#  is_vpn_x_ip_risk → 0
X_train['is_vpn_x_ip_risk'] = X_train['is_vpn_x_ip_risk'].fillna(0)
X_test['is_vpn_x_ip_risk'] = X_test['is_vpn_x_ip_risk'].fillna(0)

In [451]:
# Numériques (à scaler)
#num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes catégorielles
#onehot_cols = ["signup_source","os","browser","device_type","channel","plan_type","country"]
#target_cols = ["payment_method","merchant_category","occupation","city"]

# Colonnes texte pour TF-IDF
#text_cols = ["customer_note","last_ticket_subject"]

In [452]:
#Numériques (à scaler)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Colonnes catégorielles
onehot_cols = ["signup_source","os","browser","device_type","channel","plan_type","country"]
target_cols = ["payment_method","merchant_category","occupation","city"]

# Colonnes texte pour TF-IDF
text_cols = ["customer_note","last_ticket_subject"]

In [453]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # médiane pour credit_score, device_trust_z=0 etc
    ("scaler", RobustScaler())                      # robuste aux outliers
])

onehot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

target_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("target", TargetEncoder())
])

text_transformers = [
    (
        f"tfidf_{col}",
        TfidfVectorizer(max_features=50),
        col
    )
    for col in text_cols
]

In [454]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("onehot", onehot_pipeline, onehot_cols),
        ("target", target_pipeline, target_cols),
        *text_transformers
    ],
    remainder="drop"
)

In [455]:
X_train_enc = preprocessor.fit_transform(X_train, y_train)
X_test_enc = preprocessor.transform(X_test)

In [456]:
clf2 = LogisticRegression(
    solver="saga",       # adapté pour penalty="l1"
    penalty="l1",        # Lasso, pour faire un peu de feature selection
    class_weight={0:1, 1:24},  # ajustable selon déséquilibre
    max_iter=500,
    random_state=42
)


In [457]:
clf2.fit(X_train_enc, y_train)

c:\Users\pauld\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,"{0: 1, 1: 24}"
,random_state,42
,solver,'saga'
,max_iter,500
,multi_class,'deprecated'


In [458]:
y_pred = clf2.predict(X_test_enc)
y_proba = clf2.predict_proba(X_test_enc)[:, 1]  # probabilité de fraude

In [459]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    auc
)
import matplotlib.pyplot as plt

In [460]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.78      0.87     26671
           1       0.07      0.49      0.12       850

    accuracy                           0.77     27521
   macro avg       0.52      0.63      0.49     27521
weighted avg       0.95      0.77      0.84     27521



In [461]:
recall = recall_score(y_test, y_pred)
print("Recall (fraude) :", recall)

Recall (fraude) : 0.4894117647058824


In [462]:
precision = precision_score(y_test, y_pred)
print("Precision (fraude) :", precision)

Precision (fraude) : 0.06568766777198799


In [465]:
y_proba = clf2.predict_proba(X_test_enc)[:,1]

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall_curve, precision_curve)

print("PR-AUC :", pr_auc)

PR-AUC : 0.06927522101954961


In [ ]:
print("Recall :", recall)
print("Precision :", precision)
print("PR-AUC :", pr_auc)
print("ROC-AUC :", roc_auc)

Recall : 0.49411764705882355
Precision : 0.06629834254143646
PR-AUC : 0.07627806845736018
ROC-AUC : 0.6967622467231428


La régression logistique n'arrive pas à capter les relations, que j'enlève les features très corrélées, que je modifie le scaling
que j'encode les categorielles ou non...

## B. Feature selection basée sur la corrélation / multicolinéarité

from statsmodels.stats.outliers_influence import variance_inflation_factor

def reduce_vif(X, threshold=5.0):
    X = X.copy()
    while True:
        vif_data = pd.DataFrame()
        vif_data["feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break
        # Vérifier importance / corrélation avec y avant suppression si possible
        feature_to_drop = vif_data.sort_values("VIF", ascending=False)["feature"].iloc[0]
        print(f"Supprimer {feature_to_drop} avec VIF={max_vif:.2f}")
        X = X.drop(columns=[feature_to_drop])
    return X

X_train_reduced = reduce_vif(X_train)
X_test_reduced = X_test[X_train_reduced.columns]

# Liste des colonnes à supprimer
col_vif_trop_fort = [
    'terms_accepted_flag',
    'credit_score',
    'income_log',
    'avg_amount_30d_eur',
    'credit_score_norm',
    'income_estimate_alt_eur',
    'max_to_avg_ratio',
    'age',
    'tx_amount_total_30d_eur'
]

# Supprimer les colonnes dans X_train et X_test
X_train = X_train.drop(columns=col_vif_trop_fort)
X_test = X_test.drop(columns=col_vif_trop_fort)

# Regression logistique